# Sanity Check and Artifact Audit
Verify pre-token alignment, deterministic extraction, model provenance, and artifact reload before analysis.

In [ ]:
from pathlib import Path
import json
import numpy as np
from trajectory_extractor import RunStore

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN_ID = 'local-smoke'
store = RunStore(ROOT / 'runs')
batch = store.load_batch(RUN_ID)
runs = [store.read(RUN_ID, example_id) for example_id in batch.example_ids]
{
    'examples': len(runs),
    'shape': batch.hidden_states.shape,
    'finite': bool(np.isfinite(batch.hidden_states).all()),
    'alignments': sorted({run.provenance.get('activation_alignment') for run in runs}),
    'model_revisions': sorted({run.provenance.get('resolved_model_revision') for run in runs}),
}


In [ ]:
assert all(run.provenance.get('activation_alignment') == 'pre_response_token' for run in runs)
assert all(run.hidden_states.shape[0] == len(run.response_token_ids) for run in runs)
manifest = json.loads((ROOT / 'runs' / RUN_ID / 'manifest.json').read_text())
manifest


Stop if alignment, revision, or reload checks fail. The smoke command persists both baseline and a non-zero steering run.